# Observable Estimation

This notebook demonstrates how to compute expectation values of quantum observables using Quax. We cover:

1. **Pauli observables** on qubit systems
2. **Weyl-Heisenberg observables** on qutrit and mixed-dimension systems
3. **Bitstring probabilities** and computational-basis measurements
4. **Batched estimation** across ensembles of states and observables

The expectation value of a Hermitian observable $A$ for a quantum state is:

$$\langle A \rangle = \begin{cases} \langle\psi|A|\psi\rangle & \text{for pure states } |\psi\rangle \\ \mathrm{Tr}[A\rho] & \text{for density matrices } \rho \end{cases}$$

In Quax, the `qx.estimate(state, observable)` function handles both cases with automatic dispatch.

In [ ]:
import jax
import jax.numpy as jnp
import quax as qx
from quax.gates import I, X, Y, Z, H

## Single-qubit Pauli observables

The Pauli matrices $\{I, X, Y, Z\}$ form a complete basis for single-qubit observables. Any Hermitian $2\times 2$ matrix can be written as $A = a_0 I + a_1 X + a_2 Y + a_3 Z$ with real coefficients.

Let's compute expectations for the standard basis states:

In [ ]:
# |0> state: eigenstate of Z with eigenvalue +1
ket0 = qx.zero_state_vector(1)
print("State: |0>")
print(f"  <I> = {qx.estimate(ket0, I):.4f}")
print(f"  <X> = {qx.estimate(ket0, X):.4f}")
print(f"  <Y> = {qx.estimate(ket0, Y):.4f}")
print(f"  <Z> = {qx.estimate(ket0, Z):.4f}")

# |+> = H|0>: eigenstate of X with eigenvalue +1
ket_plus = H @ ket0
print("\nState: |+> = H|0>")
print(f"  <I> = {qx.estimate(ket_plus, I):.4f}")
print(f"  <X> = {qx.estimate(ket_plus, X):.4f}")
print(f"  <Y> = {qx.estimate(ket_plus, Y):.4f}")
print(f"  <Z> = {qx.estimate(ket_plus, Z):.4f}")

## Multi-qubit Pauli observables

For $n$-qubit systems, the Pauli basis extends to $\mathcal{P}^{\otimes n} = \{I, X, Y, Z\}^{\otimes n}$, giving $4^n$ basis operators. These are constructed using the tensor product operator `|`.

In [ ]:
# Two-qubit Bell state: |Phi+> = (|00> + |11>) / sqrt(2)
from quax.gates import CNOT

bell = CNOT @ (H | I) @ qx.zero_state_vector(2)
print("State: |Phi+> = (|00> + |11>) / sqrt(2)")
print(f"  <II> = {qx.estimate(bell, I | I):.4f}")
print(f"  <XX> = {qx.estimate(bell, X | X):.4f}")
print(f"  <YY> = {qx.estimate(bell, Y | Y):.4f}")
print(f"  <ZZ> = {qx.estimate(bell, Z | Z):.4f}")
print(f"  <ZI> = {qx.estimate(bell, Z | I):.4f}")
print(f"  <IZ> = {qx.estimate(bell, I | Z):.4f}")
print("\nNote: <XX> = <ZZ> = 1 confirms maximal entanglement")
print("      <ZI> = <IZ> = 0 confirms the reduced states are maximally mixed")

## Estimation on density matrices

The same `estimate` function works on density matrices. This is essential for noisy simulations where the state is mixed.

In [ ]:
# Pure state as density matrix
rho_pure = qx.zero_state_matrix(1)
print(f"Pure |0><0|: <Z> = {qx.estimate(rho_pure, Z):.4f}")

# Maximally mixed state: I/2
rho_mixed = qx.mixed_state_matrix(1)
print(f"Mixed I/2:    <Z> = {qx.estimate(rho_mixed, Z):.4f}")

# Apply depolarizing noise to see expectation values decay
p = 0.3
dep = qx.channels.depolarizing(jnp.array(p), dims=(2,))
rho_noisy = dep @ rho_pure
print(f"Depolarized (p={p}): <Z> = {qx.estimate(rho_noisy, Z):.4f}")
print(f"  Expected: (1-p) = {1 - p:.4f}")

## Bitstring probabilities

A complementary view of measurement is the probability of observing specific computational basis outcomes. Quax provides `qx.probabilities(state)` for the full distribution and `qx.bitstring_probability(state, bitstring)` for specific outcomes.

In [ ]:
# Full probability distribution of a 2-qubit state
psi = CNOT @ (H | I) @ qx.zero_state_vector(2)
probs = qx.probabilities(psi)
print("Bell state probabilities:")
for i, p in enumerate(probs):
    bitstring = format(i, f"0{2}b")
    print(f"  P(|{bitstring}>) = {p:.4f}")

# Specific bitstring probability
p00 = qx.bitstring_probability(psi, jnp.array([0, 0]))
p11 = qx.bitstring_probability(psi, jnp.array([1, 1]))
print(f"\nP(|00>) = {p00:.4f}")
print(f"P(|11>) = {p11:.4f}")

# Sum over a subspace
p_subspace = qx.bitstring_probability(psi, jnp.array([[0, 0], [1, 1]]))
print(f"P(|00> or |11>) = {p_subspace:.4f}")

## Computing qudit observables

### The Hermitian Weyl basis

For qubits, the four Pauli matrices $\{I, X, Y, Z\}$ form an orthogonal basis for the space of $2 \times 2$ Hermitian operators. For qudits of dimension $d > 2$, we need a larger basis: any $d \times d$ Hermitian operator lives in a $d^2$-dimensional real vector space, so we need $d^2$ basis elements.

Quax uses the **Hermitian Weyl basis**, a construction that generalises the Pauli matrices to arbitrary dimension. For $d = 2$ it reduces to the standard Paulis; for $d = 3$ (qutrits) it gives 9 basis operators that are trace-orthogonal and Hermitian. This basis is built from the **Weyl-Heisenberg displacement operators** — the clock and shift matrices that generate the discrete phase space of a qudit.

Computing expectations in this basis is exactly analogous to the qubit case: for any qutrit observable $A = \sum_i a_i W_i$ where $W_i$ are basis elements, we have $\langle A \rangle = \sum_i a_i \langle W_i \rangle$.

In [ ]:
# Build the qutrit Hermitian Weyl basis
qutrit_basis = qx.hermitian_weyl_basis(qudit_dim=3)
qutrit_labels = qx.hermitian_weyl_basis_labels(qudit_dim=3)

print(f"Qutrit basis: {len(qutrit_labels)} operators")
for label in qutrit_labels:
    print(f"  {label}")

In [ ]:
# Compute expectations for the qutrit |0> state
qutrit_0 = qx.zero_state_vector(dims=(3,))

print("Qutrit |0> expectations:")
for label, obs in zip(qutrit_labels, qutrit_basis.matrix):
    obs_obj = qx.Observable.from_matrix(obs, dims=((3,), (3,)))
    val = qx.estimate(qutrit_0, obs_obj)
    if abs(val) > 1e-10:
        print(f"  <{label}> = {val:.4f}")

In [ ]:
# Superposition state: (|0> + |1> + |2>) / sqrt(3)
superpos_data = jnp.ones(3, dtype=complex) / jnp.sqrt(3)
qutrit_superpos = qx.StateVector.from_matrix(superpos_data, dims=(3,))

print("Qutrit superposition (|0>+|1>+|2>)/sqrt(3) expectations:")
for label, obs in zip(qutrit_labels, qutrit_basis.matrix):
    obs_obj = qx.Observable.from_matrix(obs, dims=((3,), (3,)))
    val = qx.estimate(qutrit_superpos, obs_obj)
    if abs(val) > 1e-10:
        print(f"  <{label}> = {val:.4f}")

## Batched estimation over ensembles

One of Quax's strengths is efficient batch operations. We can compute expectation values for ensembles of states, ensembles of observables, or both -- all in a single call with automatic broadcasting.

This is useful for:
- Computing all Pauli expectations simultaneously
- Tracking observable evolution across a parameter sweep
- Computing expectations for multiple trajectories

In [ ]:
# Sweep RX(theta)|0> for theta in [0, 2pi] and track Pauli expectations
thetas = jnp.linspace(0, 2 * jnp.pi, 49)

# Create an ensemble of rotated states using batched scalar multiplication
# (-theta/2) * X gives an ensemble of Operators; cis() exponentiates each one
generators = (-thetas / 2) * X
psi_ensemble = qx.cis(generators) @ qx.zero_state_vector(1)

# Compute Pauli expectations for each state in the ensemble
z_vals = qx.estimate(psi_ensemble, Z)
x_vals = qx.estimate(psi_ensemble, X)
y_vals = qx.estimate(psi_ensemble, Y)

print(f"Ensemble of {len(thetas)} states")
print(f"<Z> range: [{z_vals.min():.4f}, {z_vals.max():.4f}]")
print(f"<X> range: [{x_vals.min():.4f}, {x_vals.max():.4f}]")
print(f"<Y> range: [{y_vals.min():.4f}, {y_vals.max():.4f}]")
print(f"\n<Z> at theta=0:   {z_vals[0]:.4f} (expected: 1)")
print(f"<Z> at theta=pi:   {z_vals[24]:.4f} (expected: -1)")
print(f"<Z> at theta=pi/2: {z_vals[12]:.4f} (expected: ~0)")

## Custom observables

Any Hermitian matrix can serve as an observable. Quax's `Observable` type simply wraps a Hermitian operator. You can construct observables from linear combinations of Pauli strings or directly from matrices.

In [ ]:
# A custom 2-qubit observable: the Heisenberg interaction
# H_heis = XX + YY + ZZ
heisenberg = qx.Observable.from_matrix(((X | X) + (Y | Y) + (Z | Z)).matrix, dims=((2, 2), (2, 2)))

# Product state: should have <H> = -1
product = qx.zero_state_vector(2)
print(f"Product |00>:  <XX+YY+ZZ> = {qx.estimate(product, heisenberg):.4f}")

# Bell state: should have <H> = 3 (maximally correlated)
bell = CNOT @ (H | I) @ qx.zero_state_vector(2)
print(f"Bell |Phi+>:   <XX+YY+ZZ> = {qx.estimate(bell, heisenberg):.4f}")

# Random observable
key = jax.random.key(42)
random_obs = qx.random_observable(dims=((2,), (2,)), key=key)
print(f"\nRandom 1Q observable estimated on |0>: {qx.estimate(ket0, random_obs):.4f}")

## Qutrit bitstring probabilities

Bitstring probabilities generalize naturally to qudits. For a qutrit, a 'bitstring' is a sequence of trit values $\{0, 1, 2\}$.

In [ ]:
# Two-qutrit state
psi_2qt = qx.zero_state_vector(dims=(3, 3))

# Full probability distribution
probs = qx.probabilities(psi_2qt)
print("Two-qutrit |0,0> probabilities:")
for i, p in enumerate(probs):
    t0, t1 = divmod(i, 3)
    if p > 1e-10:
        print(f"  P(|{t0},{t1}>) = {p:.4f}")

# Computational subspace probability
comp_basis = jnp.array([[0], [1]])
qutrit_state = qx.StateVector.from_matrix(jnp.array([1, 0, 0], dtype=complex) / 1.0, dims=(3,))
p_comp = qx.bitstring_probability(qutrit_state, comp_basis)
print(f"\nQutrit |0> computational subspace P(|0> or |1>) = {p_comp:.4f}")